In [ ]:
# Part 1: GPT architecture
from dataclasses import dataclass

import torch
import torch.nn as nn

@dataclass
class Config:
    n_vocab: int = 50257  #256(bytes)+50,000(BPE merges)+1(endoftext)=50,257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12

In [ ]:
#Task 2.01: Model configuration

Answer (Task 2.01):

n_vocab = 50,257 — size of the vocabulary. GPT-2 uses Byte Pair Encoding (BPE) tokenisation. The number 50,257 comes from: 50,000 BPE merge rules + 256 base byte tokens + 1 special <|endoftext|> token = 50,257 total tokens.
n_ctx = 1024 — the maximum context length (i.e. the maximum number of tokens the model can attend to at once). This is the size of the context window.
n_embd = 768 — the embedding dimension. Each token is represented as a 768-dimensional vector throughout the model.
n_head = 12 — number of attention heads in each multi-head attention layer. Each head operates on 768 / 12 = 64 dimensions.
n_layer = 12 — number of stacked Transformer decoder blocks in the model.

In [ ]:
 #Task 2.02: Mathematical properties of the GELU
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

In [ ]:
# Task 2.03: Shape annotations
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        x = self.c_fc(x)
        x = gelu(x)
        x = self.c_proj(x)
        return x

In [ ]:
#Task 2.04: Causal mask
def make_causal_mask(n):
   return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)
#test
x = torch.rand(1, 2, 3, 3)
mask = make_causal_mask(5)
x + mask[:3, :3]
print(mask)
print(x)

In [ ]:
#Task 2.05: Multi-head attention
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        head_embd = n_embd // self.n_head
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        x = q @ k.transpose(-1, -2)
        x = x / head_embd**0.5
        x = x + self.mask[:seq_len, :seq_len]
        x = torch.softmax(x, dim=-1)
        x = x @ v
        x = x.transpose(-2, -3).contiguous()
        x = x.view(batch_size, seq_len, n_embd)
        x = self.c_proj(x)
        return x

In [ ]:
#Task 2.06: Layer normalization
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd))
        self.b = nn.Parameter(torch.zeros(config.n_embd))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(unbiased=False, dim=-1, keepdim=True)
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b

In [ ]:
#Task 2.07: Pre-norm and post-norm architectures
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config)
        self.attn = Attention(config)
        self.ln_2 = LayerNorm(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
#Task 2.08: Buffers
def make_positions(n):
    return torch.arange(n, dtype=torch.long)


class Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = LayerNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

In [ ]:
#Task 2.09: Number of trainable parameters

config = Config()
model = Model(config)

n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params:,}") #after execution Trainable parameters: 163,037,184

In [ ]:
#Part 2: Load pre-trained weights
#Task 2.10: Load pre-trained weights
import numpy as np

pretrained = np.load("gpt-2-pretrained.npz")
def copy_weights(source: np.ndarray, target: torch.Tensor):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))

def from_pretrained() -> Model:
    """
    Construct a GPT-2 model and populate its parameters with OpenAI's
    pre-trained weights stored in 'gpt-2-pretrained.npz'.

    Key detail: PyTorch stores Linear layer weights transposed relative
    to the convention used in TensorFlow / the npz archive. Weights from
    the archive therefore need to be transposed before being copied.
    """
    config = Config()
    model = Model(config)
    model.lm_head.weight = model.wte.weight  # weight sharing

    pt = np.load("gpt-2-pretrained.npz")

    # --- Embeddings ---
    copy_weights(pt["wte"], model.wte.weight)       # [n_vocab, n_embd]
    copy_weights(pt["wpe"], model.wpe.weight)       # [n_ctx,   n_embd]

    # --- Transformer blocks ---
    for i in range(config.n_layer):
        blk = model.h[i]

        # Layer norm 1
        copy_weights(pt[f"h{i}.ln_1.g"],         blk.ln_1.g)
        copy_weights(pt[f"h{i}.ln_1.b"],         blk.ln_1.b)

        # Attention — note the transpose for weight matrices
        copy_weights(pt[f"h{i}.attn.c_attn.w"].T, blk.attn.c_attn.weight)
        copy_weights(pt[f"h{i}.attn.c_attn.b"],   blk.attn.c_attn.bias)
        copy_weights(pt[f"h{i}.attn.c_proj.w"].T, blk.attn.c_proj.weight)
        copy_weights(pt[f"h{i}.attn.c_proj.b"],   blk.attn.c_proj.bias)

        # Layer norm 2
        copy_weights(pt[f"h{i}.ln_2.g"],         blk.ln_2.g)
        copy_weights(pt[f"h{i}.ln_2.b"],         blk.ln_2.b)

        # MLP
        copy_weights(pt[f"h{i}.mlp.c_fc.w"].T,   blk.mlp.c_fc.weight)
        copy_weights(pt[f"h{i}.mlp.c_fc.b"],     blk.mlp.c_fc.bias)
        copy_weights(pt[f"h{i}.mlp.c_proj.w"].T, blk.mlp.c_proj.weight)
        copy_weights(pt[f"h{i}.mlp.c_proj.b"],   blk.mlp.c_proj.bias)

    # --- Final layer norm ---
    copy_weights(pt["ln_f.g"], model.ln_f.g)
    copy_weights(pt["ln_f.b"], model.ln_f.b)

    model.eval()  # disable dropout (if any) for inference
    return model

In [ ]:
# Part 3: Put the model to use
def generate(model, context, context_size=1024, n_tokens=20):
    for _ in range(n_tokens):
        context = context[:, -context_size:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        next_idx = torch.argmax(logits, dim=-1, keepdim=True)
        context = torch.cat([context, next_idx], dim=-1)
    return context

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
model = from_pretrained()


def generate_helper(text, context_size=1024, n_tokens=20):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)
    return tokenizer.decode(context[0].tolist())

print(generate_helper("Linköping University is"))

In [ ]:
#Task 2.11: Sampling-based text generation
def generate_sampled(
    model,
    context,
    context_size: int = 1024,
    n_tokens: int = 20,
    temperature: float = 1.0,
    top_k: int | None = None,
):
    """
    Sampling-based text generation with temperature scaling and top-k filtering.

    Parameters
    ----------
    model        : The language model.
    context      : Input token IDs, shape [1, T].
    context_size : Maximum context window.
    n_tokens     : Number of new tokens to generate.
    temperature  : Softmax temperature.
                   < 1.0 → sharper distribution (more deterministic).
                   > 1.0 → flatter distribution (more diverse/random).
    top_k        : If provided, restrict sampling to the top-k most probable
                   tokens at each step.

    Returns
    -------
    context : Tensor of shape [1, T + n_tokens] containing the full sequence.
    """
    for _ in range(n_tokens):
        context_trimmed = context[:, -context_size:]  # [1, ≤context_size]

        with torch.no_grad():
            logits = model(context_trimmed)[:, -1, :]  # [1, n_vocab]

        # 1. Temperature scaling
        logits = logits / temperature  # [1, n_vocab]

        # 2. Top-k filtering
        if top_k is not None:
            # Identify the k-th largest logit value
            top_k_vals, _ = torch.topk(logits, k=top_k, dim=-1)  # [1, k]
            cutoff = top_k_vals[:, -1].unsqueeze(-1)               # [1, 1]
            logits = logits.masked_fill(logits < cutoff, float("-inf"))  # [1, n_vocab]

        # 3. Convert to probabilities and sample
        probs = torch.softmax(logits, dim=-1)            # [1, n_vocab]
        next_idx = torch.multinomial(probs, num_samples=1)  # [1, 1]

        context = torch.cat([context, next_idx], dim=-1)

    return context


def sampled_helper(text, n_tokens=60, temperature=0.8, top_k=40):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate_sampled(
        model, context, n_tokens=n_tokens, temperature=temperature, top_k=top_k
    )
    return tokenizer.decode(context[0].tolist())


# Test
print("Temperature = 0.5 (focused):")
print(sampled_helper("Linköping University is", temperature=0.5, top_k=20))

print("\nTemperature = 1.0 (balanced):")
print(sampled_helper("Linköping University is", temperature=1.0, top_k=40))

print("\nTemperature = 1.5 (creative):")
print(sampled_helper("Linköping University is", temperature=1.5, top_k=50))

In [ ]:
#Task 2.12: Evaluating the pretrained model
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        sample = json.loads(line)
        prefix = tokenizer.encode(sample["ctx"])
        ending_scores = []
        for i, ending in enumerate(sample["endings"]):
            suffix = tokenizer.encode(" " + ending)
            context = torch.tensor([prefix + suffix], dtype=torch.long)
            with torch.no_grad():
                logits = model(context)
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )
            ending_scores.append((ending_score, i))
        predicted = min(ending_scores)[1]
        n_correct += int(predicted == sample["label"])
        n_total += 1
    print(f"Accuracy: {n_correct / n_total:.2%}")